In [1]:
%%file loopback_arb.py

from acadia.runtime import Runtime

class LoopbackArbRuntime(Runtime):
    
    @staticmethod
    def main():
        import time
        import numpy as np
        
        from acadia.system import Acadia, StreamConfiguration
        from acadia.channel import Channel
        from acadia.arrays import ProceduralWaveform
        from acadia.data import DataManager, ArrayRecordGroup
        
        acadia = Acadia()

        pulse_channel = acadia.DAC(1)

        def pulse_shape(out, sample_times):
            out[:] = Channel.to_samples(0.99*np.ones(len(sample_times), dtype=np.complex64))

        pulse = ProceduralWaveform(pulse_channel, generator=pulse_shape, region=pulse_channel)

        capture_channel = acadia.ADC(1)
        capture_data = ProceduralWaveform(capture_channel, generator=None, length=5000e-9, region=acadia.PLDDR0Array)
        capture_configuration = StreamConfiguration(capture_channel, acadia=acadia)

        def configure():
            pulse_channel.set_nyquist_zone(2)
            pulse_channel.configure_nco(frequency=2000e6)
            pulse_channel.set_vop(20000)
            
            capture_channel.set_nyquist_zone(2)
            capture_channel.set_dsa(0)
            
        SHOTS = 1000
            
        # We'll collect the data traces in a record group
        traces = ArrayRecordGroup((len(capture_data),), 
                                SHOTS, 
                                dtype=np.complex64, 
                                record_axes=[capture_data.axis()])
            
        # Make a data manager for storing data and serving it to a plotter
        mgr = DataManager("/home/root")

        def last_record(data):
            return {"records": np.array([np.mean(data["records"], axis=0)]), "axis0": data["axis0"]}

        mgr.add_group("traces", traces, preprocessor=last_record)
        
        # Create a sequence for the sequencer
        def sequence(a):
            capture_configuration.reset()
            for i in range(100):
                a._active_sequencer.nop()
                
            # with a.channel_synchronizer():
            #     a.generate(pulse)
            #     a.capture(capture_configuration, capture_data)
            
            with a.channel_synchronizer(block=False):
                a.capture(capture_configuration, capture_data)
                a.generate(pulse)
                a.generate(pulse)
                a.generate(pulse)
                
            # pulses = a.sequencer().DSP()
            # pulses.load(3)
                
            # with a.sequencer().repeat_until(pulses == 5):
            #     with a.sequencer().test(a.all_channel_fifos_empty(pulse_channel)):
            #         with a.channel_synchronizer(trigger=False, block=False):
            #             a.generate(pulse)
            #         pulses += 1

        # Because the pulse is procedurally generated, we can change its length at runtime,
        # so we need to start by allocating the length to use initially
        pulse.allocate(500e-9)

        # Attach to the hardware
        acadia.attach()

        # Load the wave memory with the pulse by calling the generator function
        pulse.populate()

        # Configure channel parameters using the function we defined above
        configure()

        # Configure the stream processing path to capture data using the configuration
        # written above
        acadia.configure_stream(capture_configuration)

        # Compile only once
        acadia.compile(sequence)
        
        # import logging
        # logging.basicConfig(format="[%(asctime)s] (%(threadName)s) %(levelname)s: %(message)s", level=logging.DEBUG)
        
        mgr.start_server()
        
        for shot in mgr.progress(range(SHOTS)):
            acadia.run(assemble=(shot==0))
            
            # Get the trace data and write it into a record
            trace = Channel.from_samples(capture_data.memory())
            mgr.append("traces", trace)
            
            # # Wait some time until running again just so that we can see the plot update
            # time.sleep(0.001)
                    
    # def plot(self):
    #     import matplotlib.pyplot as plt
    #     import time
    #     import numpy as np
        
    #     from acadia.data import DataManager
        
    #     fig,ax = plt.subplots(figsize=(8,4))
    #     (line_re,) = ax.plot([], [])
    #     (line_im,) = ax.plot([], [])
    #     ax.set_ylim(-0.01, 0.01)
    #     ax.set_xlim(0,5)
    #     ax.grid()
        
    #     def update(address, frame):
    #         traces = DataManager.receive_group("traces", address)
            
    #         if traces is not None:
    #             # Just plot only the most recently received trace
    #             axis = traces.axis()*1e6
    #             trace = traces.data()[-1,:]
                
    #             # Prepare the background
    #             line_re.set_data(axis, np.real(trace))
    #             line_im.set_data(axis, np.imag(trace))
    #             # ax.set_title(str( frame))
                
    #             # ax.relim()
    #             # ax.autoscale_view()
    #         return line_re, line_im
        
    #     return fig, update

Overwriting loopback_arb.py


In [2]:
%matplotlib widget
from loopback_arb import LoopbackArbRuntime
import logging

rt = LoopbackArbRuntime("192.168.2.69", "loopback_arb.py")
rt.run()

  0%|          | 0/1000 [00:00<?, ?it/s]

In [3]:
print(rt._proc.stdout.read().decode())

metal: debug:     added page size 4096 @/tmp
metal: debug:     registered platform bus
metal: debug:     registered pci bus
metal: info:      Registered shmem provider linux_shm.
metal: debug:     metal_ion_shm_provider_init: successfully initialze ion shm provider.
metal: info:      Registered shmem provider ion.reserved.
metal: info:      Registered shmem provider ion.ion_system_contig_heap.
metal: info:      Registered shmem provider ion.ion_system_heap.
metal: info:      metal_linux_dev_open: checking driver vfio-platform,1000700000.usp_rf_data_converter,uio_pdrv_genirq
metal: info:      metal_linux_dev_open: checking driver uio_pdrv_genirq,1000700000.usp_rf_data_converter,uio_pdrv_genirq
metal: info:      metal_linux_dev_open: driver uio_pdrv_genirq bound to 1000700000.usp_rf_data_converter
metal: debug:     bound device 1000700000.usp_rf_data_converter to driver uio_pdrv_genirq
metal: debug:     opened platform:1000700000.usp_rf_data_converter as /dev/uio4
metal: warning:   readi

In [3]:
rt.stop()